Handwritten Digit Recognition avec PyTorch + CNN + GUI


Imports fondamentaux

In [1]:
pip install torch torchvision matplotlib pillow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\ProgramData\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\ProgramData\anaconda3\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "c:\ProgramData\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 736, in start
    self.io_loop.start()
  File "c:\ProgramData\anaconda3\Lib\site-packa

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

Préparation des données – chargement de MNIST

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),  # Convertit une image PIL en tensor PyTorch [0,1]
    transforms.Normalize((0.1307,), (0.3081,))  # Normalisation : moyenne et écart type du dataset MNIST
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_dataset, batch_size=1000, shuffle=False)

100%|██████████| 9.91M/9.91M [00:02<00:00, 3.46MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 235kB/s]
100%|██████████| 1.65M/1.65M [00:04<00:00, 355kB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 1.42MB/s]


Définition du modèle CNN

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)   # 1 canal (NB) → 32 filtres, kernel 3x3
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)      # évite le surapprentissage
        self.fc1 = nn.Linear(9216, 128)       # Fully Connected layer (64x12x12 = 9216)
        self.fc2 = nn.Linear(128, 10)         # 10 sorties pour 0 → 9

    def forward(self, x):
        x = F.relu(self.conv1(x))           # Convolution + activation
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)              # Réduction de dimension (28x28 → 12x12)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)             # Aplatit l’image 3D en vecteur 1D
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)      # Log softmax pour la classification


Initialisation du modèle et des outils d'entraînement

In [ ]:
model = CNN()
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

Fonction d'entraînement

In [7]:
def train(model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
            print(f"Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}]  Loss: {loss.item():.6f}")

Fonction de test

In [8]:
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)
    print(f'\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)\n')


Exécution entraînement + test

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(1, 6):  # 5 époques
    train(model, device, train_loader, optimizer, epoch)
    test(model, device, test_loader)

Train Epoch: 1 [0/60000]  Loss: 2.301571
Train Epoch: 1 [6400/60000]  Loss: 0.182794
Train Epoch: 1 [12800/60000]  Loss: 0.081648
Train Epoch: 1 [19200/60000]  Loss: 0.144980
Train Epoch: 1 [25600/60000]  Loss: 0.040288
Train Epoch: 1 [32000/60000]  Loss: 0.051755
Train Epoch: 1 [38400/60000]  Loss: 0.026131
Train Epoch: 1 [44800/60000]  Loss: 0.086492
Train Epoch: 1 [51200/60000]  Loss: 0.044939
Train Epoch: 1 [57600/60000]  Loss: 0.120770

Test set: Average loss: 0.0000, Accuracy: 9869/10000 (98.69%)

Train Epoch: 2 [0/60000]  Loss: 0.016054
Train Epoch: 2 [6400/60000]  Loss: 0.003570
Train Epoch: 2 [12800/60000]  Loss: 0.067920
Train Epoch: 2 [19200/60000]  Loss: 0.004610
Train Epoch: 2 [25600/60000]  Loss: 0.010007
Train Epoch: 2 [32000/60000]  Loss: 0.004711
Train Epoch: 2 [38400/60000]  Loss: 0.033915
Train Epoch: 2 [44800/60000]  Loss: 0.012367
Train Epoch: 2 [51200/60000]  Loss: 0.002316
Train Epoch: 2 [57600/60000]  Loss: 0.010972

Test set: Average loss: 0.0000, Accuracy: 987

Sauvegarde du modèle

In [10]:
torch.save(model.state_dict(), "mnist_cnn.pth")

GUI pour dessiner et prédire

In [17]:
import tkinter as tk
from PIL import Image, ImageDraw, ImageOps
import torchvision.transforms as transforms

class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Draw a digit (0–9)")
        self.canvas = tk.Canvas(self, width=200, height=200, bg='white')
        self.canvas.pack()
        self.image = Image.new("L", (200, 200), 'white')
        self.draw = ImageDraw.Draw(self.image)

        self.canvas.bind("<B1-Motion>", self.paint)
        tk.Button(self, text="Clear", command=self.clear).pack()
        tk.Button(self, text="Predict", command=self.predict).pack()
        self.result_label = tk.Label(self, text="")
        self.result_label.pack()

    def paint(self, event):
        x, y = event.x, event.y
        r = 8
        self.canvas.create_oval(x - r, y - r, x + r, y + r, fill='black')
        self.draw.ellipse([x - r, y - r, x + r, y + r], fill='black')

    def clear(self):
        self.canvas.delete("all")
        self.draw.rectangle([0, 0, 200, 200], fill='white')
        self.result_label.config(text="")

    def predict(self):
        img = self.image.copy()
        img = ImageOps.invert(img)
        transform = transforms.Compose([
            transforms.Resize((28, 28)),
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])
        img = transform(img).unsqueeze(0)

        with torch.no_grad():
            output = model(img)
            pred = output.argmax(dim=1).item()

        self.result_label.config(text=f"Prediction: {pred}")

# Charger modèle avant
model = CNN()
model.load_state_dict(torch.load("mnist_cnn.pth"))
model.eval()

# Lancer l'application
App().mainloop()
